# GigaChess: генерация описаний ходов на T-pro (Colab)

Связывает планы ходов в текст моделью **T-pro-it-2.0-AWQ** (32B, Т-Банк, Apache-2.0) через vLLM.
Наш код (`s5_polish.py` + валидатор) работает с vLLM как с обычным OpenAI-совместимым API.

**Перед запуском:** Среда выполнения → Сменить среду → **GPU: L4 или A100** (T4 не подойдёт: мало памяти).

1. Положите `gigachess_bundle.zip` в Google Диск, в папку `gigachess` (`Мой диск/gigachess/`).
2. Запускайте ячейки по порядку. Все результаты пишутся на Диск — если Colab отключится,
   просто запустите ячейки заново: генерация продолжится с места остановки.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile
WORK = '/content/drive/MyDrive/gigachess'
os.makedirs(WORK, exist_ok=True)
if not os.path.exists(f'{WORK}/s5_polish.py'):
    zipfile.ZipFile(f'{WORK}/gigachess_bundle.zip').extractall(WORK)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# ~3–5 минут
!pip install -q vllm python-chess==1.11.2

## Запуск сервера модели
Первый раз модель скачивается с Hugging Face (~19 ГБ, 5–10 минут), потом стартует сервер.

In [ ]:
import subprocess, time, urllib.request
MODEL = 't-tech/T-pro-it-2.0-AWQ'
log = open('/content/vllm.log', 'w')
server = subprocess.Popen(
    ['vllm', 'serve', MODEL, '--served-model-name', MODEL,
     '--max-model-len', '4096', '--gpu-memory-utilization', '0.92', '--max-num-seqs', '32'],
    stdout=log, stderr=subprocess.STDOUT)
for i in range(180):
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5)
        print('сервер готов'); break
    except Exception:
        time.sleep(10)
else:
    print('сервер не поднялся — смотрите /content/vllm.log')
!tail -5 /content/vllm.log

In [ ]:
import os, json
os.environ.update({
    'LLM_BASE_URL': 'http://localhost:8000/v1',
    'LLM_MODEL': 't-tech/T-pro-it-2.0-AWQ',
    'LLM_API_KEY': 'EMPTY',
    'LLM_MAX_TOKENS': '1000',
    'LLM_TIMEOUT': '300',
    # T-pro 2.0 — гибрид с рассуждениями: для склейки текста рассуждения выключаем.
    'LLM_EXTRA_BODY': json.dumps({'chat_template_kwargs': {'enable_thinking': False}, 'temperature': 0.3}),
    'PYTHONIOENCODING': 'utf-8',
})

## Проба на 20 примерах
Сколько проходит валидатор и как выглядят тексты. Если «принято» меньше ~17 из 20 — остановитесь и напишите.

In [ ]:
import json
done = {json.loads(l)['id'] for l in open('work_ds/desc3000_polished.jsonl', encoding='utf-8')}
rest = [l for l in open('work_ds/desc3000_drafts.jsonl', encoding='utf-8') if json.loads(l)['id'] not in done]
open('colab_try20_drafts.jsonl', 'w', encoding='utf-8').writelines(rest[:20])
print('осталось сгенерировать:', len(rest))
!rm -f colab_try20_out.jsonl colab_try20_out.rejected.jsonl
!python s5_polish.py --inp colab_try20_drafts.jsonl --out colab_try20_out.jsonl --concurrency 8 --fewshot work_local/fewshot.json 2>&1 | tail -2
for l in list(open('colab_try20_out.jsonl', encoding='utf-8'))[:5]:
    r = json.loads(l); print('•', r.get('answer'), '\n')

## Полная генерация остатка
Пишет прямо в `work_ds/desc3000_polished.jsonl` на Диске; повторный запуск продолжает с места остановки.

In [ ]:
!python s5_polish.py --inp work_ds/desc3000_drafts.jsonl --out work_ds/desc3000_polished.jsonl --concurrency 24 --fewshot work_local/fewshot.json 2>&1 | grep -v '^\[' | tail -5

In [ ]:
import json, collections
rows = [json.loads(l) for l in open('work_ds/desc3000_polished.jsonl', encoding='utf-8')]
ok = [r for r in rows if r.get('answer')]
print('готово:', len(ok), '| окончательно отклонено:', len(rows) - len(ok), '| всего планов: 3000')
print(collections.Counter(r.get('model') for r in ok))

## Дальше
Скачайте с Диска `gigachess/work_ds/desc3000_polished.jsonl` (и `desc3000_polished.rejected.jsonl`)
и положите в `gigachess_data/work_ds/` на компьютере, заменив старые. Перепроверку движком,
судью и выгрузку датасета запустит Claude локально.